# Synthetic camera trajectories — data and evaluator checks

Checks the runtime, runs the unit tests, builds every Phase-1 split, and runs the GT audit.
**Nothing here is a model result.** The *oracle* rows are GT re-draws scored under the frozen evaluation contract; they give the finite-K noise floor that model numbers will be compared against.

Runtime: CPU is enough (GPU not needed until Phase 2).

## 1. Get the code
Either upload the project ZIP when prompted, or set `REPO_URL` to your Git remote.

In [ ]:
import os, sys, glob, subprocess
REPO_URL = ""  # Optional Git remote URL for Colab; otherwise upload the repository ZIP.
IN_COLAB = "google.colab" in sys.modules

if not os.path.isdir("camera_motion"):
    candidates = sorted(os.path.dirname(p) for p in glob.glob("*/camera_motion") if os.path.isdir(p))
    if not candidates and IN_COLAB:
        if REPO_URL:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "camera-motion-flow-matching"], check=True)
        else:
            from google.colab import files
            from zipfile import ZipFile
            uploaded = files.upload()
            if not uploaded:
                raise RuntimeError("Upload the repository ZIP or specify REPO_URL.")
            with ZipFile(next(iter(uploaded))) as zf:
                zf.extractall(".")
        candidates = sorted(os.path.dirname(p) for p in glob.glob("*/camera_motion") if os.path.isdir(p))
    if candidates:
        os.chdir(candidates[0])
assert os.path.isdir("camera_motion"), f"Cannot find repository root: {os.getcwd()}"
sys.path.insert(0, os.getcwd())
print("repository root:", os.getcwd())

print("project root:", os.getcwd())

## 2. Environment

In [ ]:
import platform, numpy, scipy, matplotlib
print("python", platform.python_version(), "| numpy", numpy.__version__,
      "| scipy", scipy.__version__, "| matplotlib", matplotlib.__version__)
try:
    import torch
    print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("torch not installed (only needed from Phase 2)")

## 3. Unit tests

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(r.stdout[-1500:]); print(r.stderr[-500:])
assert r.returncode == 0, "unit tests failed - stop here and report the output"

## 4. Build all splits + GT audit

In [ ]:
from scripts.run_gt_audit import main as run_audit
report = run_audit("outputs/phase1", k_oracle=200)

## 5. Gate checks
Each line must be `True` before Phase 2 starts.

In [ ]:
import numpy as np
gate = {
  "E1a GT audit agreement == 1":       report["E1a_gt_audit"]["agreement"] == 1.0,
  "E1b GT audit agreement == 1":       report["E1b_gt_audit"]["agreement"] == 1.0,
  "GT unassigned == 0":                report["E1a_gt_audit"]["unassigned_rate"] == 0 and report["E1b_gt_audit"]["unassigned_rate"] == 0,
  "rho margin > 2x threshold":         min(report[f"{f}_gt_audit"]["rho_margin_min_abs"] for f in ("E1a","E1b")) > 2*0.08,
  "azimuth step << aliasing limit":    max(report[f"{f}_gt_audit"]["max_azimuth_step_deg"] for f in ("E1a","E1b")) < 60,
  "audit counts exactly 42/28/18/12":  all(v["generator_counts"] == [42,28,18,12] for f in ("E1a","E1b") for v in report[f"{f}_gt_audit"]["per_condition"].values()),
  "train: 1 path per condition":       report["splits"]["E1a_train"]["n_conditions"] == report["splits"]["E1a_train"]["n_paths"],
  "train mode/condition chi2 p > 1e-3": min(v["p_value"] for f in ("E1a","E1b") for v in report[f"{f}_train_independence_chi2"].values()) > 1e-3,
  "split condition IDs disjoint":      report["condition_ids_disjoint"],
  "E1b endpoints have zero offset":    report["E1b_excursion_generator_audit"]["endpoint_offset_deg_max"] < 1e-6,
  "E2 scope guard":                    report["E2_scope_guard"].startswith("ok"),
  "classifier invariances exact":      all(all(report["classifier_robustness"][f]["invariance"].values()) for f in ("E1a","E1b")) and report["classifier_robustness"]["temporal_resampling_L64_invariant"],
  "wrong-mode rate <= 0.5% under perturbation": max(report["classifier_robustness"][f]["max_wrong_rate_over_all_curves"] for f in ("E1a","E1b")) <= 0.005,
}
for k, v in gate.items(): print(f"{'PASS' if v else 'FAIL'}  {k}")
print("\nPHASE-1 GATE:", "PASS" if all(gate.values()) else "FAIL")

## 6. Reference numbers (GT only — not model results)

In [ ]:
for k in [k for k in report if k.startswith("oracle_contract")]:
    p = report[k]["pooled"]
    print(f"{k:42s} TV5 {p['tv5_mean']:.3f}  95% CI {np.round(p['tv5_ci95'],3).tolist()}  "
          f"coverage {p['coverage4_mean']:.2f}/4  unassigned {p['unassigned_rate_mean']:.3f}")
print("\nE1b W1 noise floor between two independent GT sets (K=200):")
for k, v in report["E1b_w1_noise_floor_K200"].items(): print(f"  {k:24s} {v:.3f}")
print("\nModel metrics: not available yet (Phase 2).")

## 7. Figures

In [ ]:
from IPython.display import Image, display
for f in sorted(glob.glob("outputs/phase1/figures/*.png")):
    print(os.path.basename(f)); display(Image(filename=f))

## 8. (Optional) keep the outputs
Download the report, or copy `outputs/phase1` to Drive.

In [ ]:
if IN_COLAB:
    subprocess.run(["zip", "-q", "-r", "phase1_outputs.zip", "outputs/phase1/phase1_audit_report.json",
                    "outputs/phase1/figures", "outputs/phase1/data/config_frozen.json"], check=True)
    from google.colab import files
    files.download("phase1_outputs.zip")